# Estatísticas do artigo (essencial)

Backup: `7_stats.ipynb.bak_full`. Default = primary `36m_6m` · svm · vol · combat=False.

Alinha com `6_results`: **A** = âncora metodológica (abs); **B** = claim longitudinal (deltas no gradiente).

| # | Pergunta | Papel |
|---|----------|-------|
| 1 | Idade/sexo diferem entre sMCI e pMCI? | A · demografia |
| 2 | Idade/sexo sozinhos discriminam? | A · confound |
| 3 | **abs vs t1_only** (FDR, 5 mods) | **A · teste formal** |
| 4 | deltas vs t1 (+ vs abs) no primary | B · exploratório (ganho no gradiente em `6_results` §3) |
| 5 | Clínico / imagem / fusão | A |
| 6 | abs vs leaky (global) | A · controlo |
| 7 | Tabela resumo (display) | — |

Demografia **só no primary**. Não crownear deltas como primary endpoint.


In [ ]:
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

_MOD = Path.cwd() / "modules"
if str(_MOD) not in sys.path:
    sys.path.insert(0, str(_MOD))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy import stats
from sklearn.metrics import roc_auc_score

from ablation_analysis import (
    explode_patient_predictions,
    prepare_ablation_df,
)
from stats_compare import (
    apply_bh_fdr,
    clinical_results_path,
    compare_modalities,
    fusion_results_path,
    image_ablation_path,
    paired_comparison_row,
    print_comparison_summary,
)

BASE = Path("csvs/cohorts/36m_6m")
LONG_CSV = BASE / "adnimerged_longitudinal.csv"
MODS_COMPARE = ("vol", "shape", "texture", "disp", "all")
FEATURE_MIN_COVERAGE = 50  # % folds com atributo (paper: top estáveis)


@dataclass(frozen=True)
class StatsCfg:
    task: str = "smci_pmci"
    groups: tuple[str, str] = ("sMCI", "pMCI")
    positive_group: str = "pMCI"
    modality: str = "vol"
    model_key: str = "svm"
    with_combat: bool = False
    selection_mode: str = "l1_stable"
    protocol: str = "abs"
    n_perm: int = 5000
    n_bootstrap: int = 5000
    seed: int = 42


ALPHA = 0.05  # limiar usual em artigos (5%)


def ler_p(p: float, *, contexto: str = "") -> str:
    """Texto curto para interpretar um p-value."""
    tag = "significativo" if p < ALPHA else "não significativo"
    acaso = "rejeitamos hipótese nula" if p < ALPHA else "compatível com acaso"
    prefix = f"{contexto}: " if contexto else ""
    return f"{prefix}{tag} (p={p:.4f}) — {acaso}"


STATS_CFG = StatsCfg()
print("Configuração atual:")
print(f"  base={BASE}")
print(f"  task={STATS_CFG.task}  grupos={STATS_CFG.groups}")
print(
    f"  imagem: protocolo={STATS_CFG.protocol}  modality={STATS_CFG.modality}  "
    f"model={STATS_CFG.model_key}  combat={STATS_CFG.with_combat}"
)
print(f"  permutações={STATS_CFG.n_perm}  bootstrap={STATS_CFG.n_bootstrap}")


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def load_cohort(long_path: Path, cfg: StatsCfg) -> pd.DataFrame:
    """Uma linha por paciente: slot baseline (legado) ou t0 (cohorts atuais)."""
    sub = pd.read_csv(long_path)
    slots = sub["slot"].astype(str)
    sub = sub[slots.isin(("baseline", "t0"))]
    sub = sub.sort_values(["ID_PT", "ID_IMG"]).groupby("ID_PT", as_index=False).first()
    sub = sub[sub["GROUP"].isin(cfg.groups)].copy()
    sub["ID_PT"] = sub["ID_PT"].astype(str)
    sub["y"] = (sub["GROUP"] == cfg.positive_group).astype(int)
    sub["SEX_bin"] = sub["SEX"].map({"M": 0, "F": 1, 0: 0, 1: 1})
    return sub


def permutation_auc_p(
    y: np.ndarray,
    scores: np.ndarray,
    *,
    n_perm: int,
    seed: int,
    bidirectional: bool = False,
) -> tuple[float, float]:
    y = np.asarray(y, dtype=int)
    s = np.asarray(scores, dtype=float)
    if bidirectional:
        auc_obs = float(max(roc_auc_score(y, s), roc_auc_score(y, -s)))
    else:
        auc_obs = float(roc_auc_score(y, s))
    prng = np.random.default_rng(seed)
    ge = 0
    for _ in range(n_perm):
        yp = prng.permutation(y)
        auc_n = max(roc_auc_score(yp, s), roc_auc_score(yp, -s)) if bidirectional else roc_auc_score(yp, s)
        ge += int(auc_n >= auc_obs)
    return auc_obs, (ge + 1) / (n_perm + 1)


def bootstrap_auc_diff(y, scores_a, scores_b, *, n_boot: int, seed: int) -> tuple[float, float, float]:
    y = np.asarray(y, dtype=int)
    a, b = np.asarray(scores_a, float), np.asarray(scores_b, float)
    obs = float(roc_auc_score(y, a) - roc_auc_score(y, b))
    prng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        idx = prng.integers(0, len(y), size=len(y))
        yb, ab, bb = y[idx], a[idx], b[idx]
        if len(np.unique(yb)) < 2:
            continue
        diffs.append(float(roc_auc_score(yb, ab) - roc_auc_score(yb, bb)))
    diffs = np.asarray(diffs)
    ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])
    return obs, float(ci_lo), float(ci_hi)


def bootstrap_auc_diff_test(
    y,
    scores_a,
    scores_b,
    *,
    n_boot: int,
    seed: int,
) -> tuple[float, float, float, float, float]:
    """ΔAUC = AUC(a)−AUC(b); IC95%; p one-sided (a>b) e two-sided."""
    y = np.asarray(y, dtype=int)
    a, b = np.asarray(scores_a, float), np.asarray(scores_b, float)
    obs = float(roc_auc_score(y, a) - roc_auc_score(y, b))
    prng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        idx = prng.integers(0, len(y), size=len(y))
        yb, ab, bb = y[idx], a[idx], b[idx]
        if len(np.unique(yb)) < 2:
            continue
        diffs.append(float(roc_auc_score(yb, ab) - roc_auc_score(yb, bb)))
    diffs = np.asarray(diffs)
    ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])
    p_one = float((np.sum(diffs <= 0) + 1) / (len(diffs) + 1))
    p_two = float(2 * min(p_one, 1 - p_one))
    return obs, float(ci_lo), float(ci_hi), p_one, p_two


def patient_scores_for_protocol(path: Path, cfg: StatsCfg) -> pd.DataFrame:
    return patient_image_scores(path, cfg).rename(columns={"score_img": "score"})


def nested_cv_auc_univariate(
    X: np.ndarray,
    y: np.ndarray,
    *,
    seed: int,
    k: int = 5,
) -> tuple[float, np.ndarray]:
    """LogReg + scaler em 1 atributo; AUC com predições out-of-fold."""
    X = np.asarray(X, dtype=float).reshape(-1, 1)
    y = np.asarray(y, dtype=int)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, random_state=seed)),
    ])
    cv = StratifiedKFold(k, shuffle=True, random_state=seed)
    scores = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]
    return float(roc_auc_score(y, scores)), scores


def patient_image_scores(path: Path, cfg: StatsCfg) -> pd.DataFrame:
    raw = prepare_ablation_df(pd.read_csv(path))
    mask = (
        (raw["task"] == cfg.task)
        & (raw["modality"] == cfg.modality)
        & (raw["model_key"] == cfg.model_key)
        & (raw["with_combat"] == cfg.with_combat)
        & (raw["selection_mode"] == cfg.selection_mode)
    )
    sub = raw.loc[mask]
    if sub.empty:
        raise FileNotFoundError(f"sem linhas para {cfg} em {path}")
    pat = explode_patient_predictions(sub)
    return pat.groupby("ID_PT", as_index=False).agg(y=("y", "first"), score_img=("score", "mean"))


def patient_scores_from_path(path: Path, cfg: StatsCfg) -> pd.DataFrame:
    return patient_scores_for_protocol(path, cfg)


def cfg_for_modality(mod: str, cfg: StatsCfg | None = None) -> StatsCfg:
    c = cfg or STATS_CFG
    return StatsCfg(
        task=c.task,
        groups=c.groups,
        positive_group=c.positive_group,
        modality=mod,
        model_key=c.model_key,
        with_combat=c.with_combat,
        selection_mode=c.selection_mode,
        protocol=c.protocol,
        n_perm=c.n_perm,
        n_bootstrap=c.n_bootstrap,
        seed=c.seed,
    )


def cfg_clinical(cfg: StatsCfg | None = None) -> StatsCfg:
    c = cfg or STATS_CFG
    return StatsCfg(
        task=c.task,
        groups=c.groups,
        positive_group=c.positive_group,
        modality="clinical",
        model_key=c.model_key,
        with_combat=False,
        selection_mode="none",
        protocol="clinical",
        n_perm=c.n_perm,
        n_bootstrap=c.n_bootstrap,
        seed=c.seed,
    )


cohort = load_cohort(LONG_CSV, STATS_CFG)
g0, g1 = STATS_CFG.groups
print(f"n={len(cohort)} | {cohort['GROUP'].value_counts().to_dict()}")

## 1. Idade / sexo entre grupos (âncora A · primary)


In [ ]:
age_a = cohort.loc[cohort["GROUP"] == g0, "AGE"].dropna()
age_b = cohort.loc[cohort["GROUP"] == g1, "AGE"].dropna()
_, p_age_groups = stats.mannwhitneyu(age_a, age_b, alternative="two-sided")

sex_tab = pd.crosstab(cohort["GROUP"], cohort["SEX_bin"], rownames=["GROUP"], colnames=["SEX (0=M, 1=F)"])
_, p_sex_groups, _, _ = stats.chi2_contingency(sex_tab)

demo = pd.DataFrame([
    {
        "variável": "idade",
        "teste": "Mann-Whitney U",
        "grupo_a": g0,
        "valor_a": age_a.mean(),
        "o_que_e_valor": "média idade (anos)",
        "grupo_b": g1,
        "valor_b": age_b.mean(),
        "p": p_age_groups,
    },
    {
        "variável": "sexo",
        "teste": "χ²",
        "grupo_a": g0,
        "valor_a": cohort.loc[cohort["GROUP"] == g0, "SEX_bin"].mean(),
        "o_que_e_valor": "proporção mulheres (F=1)",
        "grupo_b": g1,
        "valor_b": cohort.loc[cohort["GROUP"] == g1, "SEX_bin"].mean(),
        "p": p_sex_groups,
    },
])

print("Tabela de contagens (entrada do χ²):")
display(sex_tab)
print("\nResumo (valores descritivos + p do teste):")
display(demo.round(4))

print("\n── Interpretação ──")
print(ler_p(p_age_groups, contexto="Idade"))
print(ler_p(p_sex_groups, contexto="Sexo"))
if p_age_groups >= ALPHA and p_sex_groups >= ALPHA:
    print("✓ Grupos demograficamente parecidos → idade/sexo entre classes pouco provável que confundam.")
else:
    print("⚠ Pelo menos uma variável difere → interpretar modelos da secção 2 com cautela.")

## 2. Idade ou sexo sozinhos discriminam? (âncora A)


In [ ]:
cfg = STATS_CFG
pt = cohort[["ID_PT", "y", "AGE", "SEX_bin"]].dropna().copy()
y = pt["y"].to_numpy()

print("Passo A — modelo só com IDADE (nested CV + permutação)")
auc_age, scores_age = nested_cv_auc_univariate(pt["AGE"].to_numpy(), y, seed=cfg.seed)
_, p_age = permutation_auc_p(y, scores_age, n_perm=cfg.n_perm, seed=cfg.seed)
print(f"  AUC = {auc_age:.3f}  |  {ler_p(p_age)}")

print("\nPasso B — modelo só com SEXO")
auc_sex, scores_sex = nested_cv_auc_univariate(pt["SEX_bin"].to_numpy(), y, seed=cfg.seed + 1)
_, p_sex = permutation_auc_p(y, scores_sex, n_perm=cfg.n_perm, seed=cfg.seed + 1)
print(f"  AUC = {auc_sex:.3f}  |  {ler_p(p_sex)}")

pt = pt.assign(score_age=scores_age, score_sex=scores_sex)

print(f"\nPasso C — scores de IMAGEM ({cfg.protocol} / {cfg.modality})")
results_path = image_ablation_path(BASE, cfg.protocol, cfg.modality)
img = patient_image_scores(results_path, cfg).merge(pt, on=["ID_PT", "y"], how="inner")
y_m = img["y"].to_numpy()

auc_img, p_img = permutation_auc_p(
    y_m, img["score_img"].to_numpy(), n_perm=cfg.n_perm, seed=cfg.seed + 2,
)
print(f"  AUC = {auc_img:.3f}  |  {ler_p(p_img)}  |  n pacientes = {len(img)}")

print("\nPasso D — imagem melhor que demografia? (bootstrap do ΔAUC)")
d_age, ci_age_lo, ci_age_hi = bootstrap_auc_diff(
    y_m, img["score_img"].to_numpy(), img["score_age"].to_numpy(),
    n_boot=cfg.n_bootstrap, seed=cfg.seed + 3,
)
d_sex, ci_sex_lo, ci_sex_hi = bootstrap_auc_diff(
    y_m, img["score_img"].to_numpy(), img["score_sex"].to_numpy(),
    n_boot=cfg.n_bootstrap, seed=cfg.seed + 4,
)
print(f"  ΔAUC (imagem − idade) = {d_age:.3f}  IC95% [{ci_age_lo:.3f}, {ci_age_hi:.3f}]")
print(f"  ΔAUC (imagem − sexo)  = {d_sex:.3f}  IC95% [{ci_sex_lo:.3f}, {ci_sex_hi:.3f}]")

summary = pd.DataFrame([
    {"modelo": "só idade", "features": "AGE", "auc": auc_age, "p_permutation": p_age},
    {"modelo": "só sexo", "features": "SEX_bin", "auc": auc_sex, "p_permutation": p_sex},
    {"modelo": f"imagem ({cfg.protocol})", "features": cfg.modality, "auc": auc_img, "p_permutation": p_img},
    {"modelo": "imagem − idade", "features": "ΔAUC", "auc": d_age, "p_permutation": np.nan},
    {"modelo": "imagem − sexo", "features": "ΔAUC", "auc": d_sex, "p_permutation": np.nan},
])
summary["ci95"] = ["", "", "", f"[{ci_age_lo:.3f}, {ci_age_hi:.3f}]", f"[{ci_sex_lo:.3f}, {ci_sex_hi:.3f}]"]
summary["confunde"] = [
    bool(auc_age > 0.55 and p_age < ALPHA),
    bool(auc_sex > 0.55 and p_sex < ALPHA),
    False,
    np.nan,
    np.nan,
]
summary["imagem_superior"] = [np.nan, np.nan, np.nan, ci_age_lo > 0, ci_sex_lo > 0]

print("\n── Tabela resumo ──")
display(summary.round(4))

print("\n── Conclusão automática ──")
if not summary.loc[0, "confunde"] and not summary.loc[1, "confunde"]:
    print("• Idade e sexo **não** separam bem as classes → pouco risco de confusão demográfica.")
else:
    print("• Atenção: idade ou sexo mostram sinal discriminativo → interpretar imagem com cuidado.")

if p_img < ALPHA and auc_img > 0.55:
    print(f"• Imagem ({cfg.protocol}) discrimina significativamente (AUC ≈ {auc_img:.2f}).")
if ci_age_lo > 0:
    print(f"• Imagem supera idade: ganho médio ΔAUC ≈ {d_age:.2f} (IC95% não cruza zero).")
if ci_sex_lo > 0:
    print(f"• Imagem supera sexo: ganho médio ΔAUC ≈ {d_sex:.2f} (IC95% não cruza zero).")



## 3. abs vs t1_only — teste formal (âncora A · FDR)

Única comparação de encoding **pré-especificada com FDR** no primary. Não substitui o claim longitudinal (`6_results` §3).


In [ ]:
cfg = STATS_CFG

cmp_abs_t1 = compare_modalities(
    MODS_COMPARE,
    path_a=lambda m: image_ablation_path(BASE, "abs", m),
    path_b=lambda m: image_ablation_path(BASE, "t1_only", m),
    cfg_for_mod=lambda m: cfg_for_modality(m, cfg),
    load_patients=patient_scores_from_path,
    permutation_auc_p=permutation_auc_p,
    n_perm=cfg.n_perm,
    n_bootstrap=cfg.n_bootstrap,
    seed=cfg.seed,
    label_a="abs",
    label_b="t1_only",
    comparison="abs_vs_t1_only",
    alpha=ALPHA,
)
print(
    f"Âncora A · teste formal | abs vs t1_only | {cfg.task} | {cfg.model_key} | "
    f"combat={cfg.with_combat} | {cfg.selection_mode}\n"
)
display(cmp_abs_t1.round(4))
print("\n── Interpretação (ΔAUC = abs − t1_only) ──")
print_comparison_summary(cmp_abs_t1, label_a="abs", label_b="t1_only")


## 4. deltas vs t1_only (e vs abs) — exploratório (claim B)

No primary (`36m_6m`, gap curto) espera-se abs ≈ t1 ≈ deltas. **Não** crownear deltas aqui.

Claim longitudinal (deltas melhora com `t_imagens`) → figura-chave em `6_results` §3 (descritivo multi-cohort, sem FDR entre cohorts).


In [ ]:
cfg = STATS_CFG

cmp_deltas_t1 = compare_modalities(
    MODS_COMPARE,
    path_a=lambda m: image_ablation_path(BASE, "deltas", m),
    path_b=lambda m: image_ablation_path(BASE, "t1_only", m),
    cfg_for_mod=lambda m: cfg_for_modality(m, cfg),
    load_patients=patient_scores_from_path,
    permutation_auc_p=permutation_auc_p,
    n_perm=cfg.n_perm,
    n_bootstrap=cfg.n_bootstrap,
    seed=cfg.seed,
    label_a="deltas",
    label_b="t1_only",
    comparison="deltas_vs_t1_only",
    alpha=ALPHA,
)
print(
    f"Claim B · exploratório | deltas vs t1_only | {cfg.task} | {cfg.model_key} | "
    f"combat={cfg.with_combat} | {cfg.selection_mode}\n"
)
display(cmp_deltas_t1.round(4))
print("\n── Interpretação (ΔAUC = deltas − t1_only) ──")
print_comparison_summary(cmp_deltas_t1, label_a="deltas", label_b="t1_only")

cmp_deltas_abs = compare_modalities(
    MODS_COMPARE,
    path_a=lambda m: image_ablation_path(BASE, "deltas", m),
    path_b=lambda m: image_ablation_path(BASE, "abs", m),
    cfg_for_mod=lambda m: cfg_for_modality(m, cfg),
    load_patients=patient_scores_from_path,
    permutation_auc_p=permutation_auc_p,
    n_perm=cfg.n_perm,
    n_bootstrap=cfg.n_bootstrap,
    seed=cfg.seed,
    label_a="deltas",
    label_b="abs",
    comparison="deltas_vs_abs",
    alpha=ALPHA,
)
print(
    f"\nClaim B · exploratório | deltas vs abs | {cfg.task} | {cfg.model_key}\n"
)
display(cmp_deltas_abs.round(4))
print("\n── Interpretação (ΔAUC = deltas − abs) ──")
print_comparison_summary(cmp_deltas_abs, label_a="deltas", label_b="abs")



## 5. Clínico vs imagem vs fusão (âncora A)


In [ ]:
cfg = STATS_CFG
mod = cfg.modality
img_cfg = cfg_for_modality(mod, cfg)
clin_cfg = cfg_clinical(cfg)
fusion_path = fusion_results_path(
    BASE, mod, selection_mode=cfg.selection_mode, with_combat=cfg.with_combat,
)
clin_path = clinical_results_path(BASE)
img_path = image_ablation_path(BASE, "abs", mod)

clinical_rows: list[dict] = []
if img_path.exists() and clin_path.exists():
    paired = patient_scores_from_path(img_path, img_cfg).merge(
        patient_scores_from_path(clin_path, clin_cfg),
        on=["ID_PT", "y"],
        suffixes=("_img", "_clin"),
    )
    row = paired_comparison_row(
        paired,
        score_a="score_img",
        score_b="score_clin",
        label_a="img",
        label_b="clin",
        n_boot=cfg.n_bootstrap,
        seed=cfg.seed + 200,
        permutation_auc_p=permutation_auc_p,
        n_perm=cfg.n_perm,
        alpha=ALPHA,
    )
    row.update({"modality": mod, "comparison": "image_abs_vs_clinical"})
    clinical_rows.append(row)

if fusion_path.exists() and clin_path.exists():
    paired = patient_scores_from_path(fusion_path, img_cfg).merge(
        patient_scores_from_path(clin_path, clin_cfg),
        on=["ID_PT", "y"],
        suffixes=("_fusion", "_clin"),
    )
    row = paired_comparison_row(
        paired,
        score_a="score_fusion",
        score_b="score_clin",
        label_a="fusion",
        label_b="clin",
        n_boot=cfg.n_bootstrap,
        seed=cfg.seed + 201,
        permutation_auc_p=permutation_auc_p,
        n_perm=cfg.n_perm,
        alpha=ALPHA,
    )
    row.update({"modality": mod, "comparison": "fusion_vs_clinical"})
    clinical_rows.append(row)

if fusion_path.exists() and img_path.exists():
    paired = patient_scores_from_path(fusion_path, img_cfg).merge(
        patient_scores_from_path(img_path, img_cfg),
        on=["ID_PT", "y"],
        suffixes=("_fusion", "_img"),
    )
    row = paired_comparison_row(
        paired,
        score_a="score_fusion",
        score_b="score_img",
        label_a="fusion",
        label_b="img",
        n_boot=cfg.n_bootstrap,
        seed=cfg.seed + 202,
        permutation_auc_p=permutation_auc_p,
        n_perm=cfg.n_perm,
        alpha=ALPHA,
    )
    row.update({"modality": mod, "comparison": "fusion_vs_image_abs"})
    clinical_rows.append(row)

cmp_clinical = pd.DataFrame(clinical_rows)
if not cmp_clinical.empty:
    cmp_clinical["p_fdr_bh"] = apply_bh_fdr(cmp_clinical["p_bootstrap_one_sided"].to_numpy())
    cmp_clinical["significant_fdr"] = (
        cmp_clinical["p_fdr_bh"] < ALPHA
    ) & (cmp_clinical["ci95_lo"] > 0)

print(f"Clínico / imagem / fusão | {cfg.task} | mod={mod} | {cfg.model_key}\n")
display(cmp_clinical.round(4))

print("\n── Interpretação ──")
for _, r in cmp_clinical.iterrows():
    print(
        f"  {r['comparison']}: ΔAUC={r['delta_auc']:.3f} "
        f"[{r['ci95_lo']:.3f}, {r['ci95_hi']:.3f}]  p={r['p_bootstrap_one_sided']:.4f}"
    )

if not cmp_clinical.empty:
    fig, ax = plt.subplots(figsize=(6, 3.5))
    labels = cmp_clinical["comparison"].str.replace("_", " ", regex=False)
    x = np.arange(len(cmp_clinical))
    ax.bar(x, cmp_clinical["delta_auc"], color="steelblue", alpha=0.85)
    ax.errorbar(
        x,
        cmp_clinical["delta_auc"],
        yerr=[
            cmp_clinical["delta_auc"] - cmp_clinical["ci95_lo"],
            cmp_clinical["ci95_hi"] - cmp_clinical["delta_auc"],
        ],
        fmt="none",
        color="0.2",
        capsize=4,
    )


## 6. abs vs global / leaky (âncora A · controlo)


In [ ]:
cfg = STATS_CFG

cmp_abs_global = compare_modalities(
    MODS_COMPARE,
    path_a=lambda m: image_ablation_path(BASE, "abs", m),
    path_b=lambda m: image_ablation_path(BASE, "global", m),
    cfg_for_mod=lambda m: cfg_for_modality(m, cfg),
    load_patients=patient_scores_from_path,
    permutation_auc_p=permutation_auc_p,
    n_perm=cfg.n_perm,
    n_bootstrap=cfg.n_bootstrap,
    seed=cfg.seed + 100,
    label_a="abs",
    label_b="global",
    comparison="abs_vs_global",
    alpha=ALPHA,
)
print(f"abs vs global (leaky) | {cfg.task} | {cfg.model_key}\n")
display(cmp_abs_global.round(4))
print("\n── Interpretação (ΔAUC = abs − global; negativo → leaky maior) ──")
print_comparison_summary(cmp_abs_global, label_a="abs", label_b="global")